# 01 - EDA: BUSI, Kvasir, MoNuSAC

The three datasets the project compares, side by side. Reads the MedSegBench `.npz`
files directly (no model, no GPU), so the whole notebook runs in under a minute.

Covers sample size, image size, class imbalance, per-image target-size distribution,
object count, and correlation between image-level properties.

Answers the report's "What data did you use?" section.


In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from pathlib import Path
from scipy import ndimage

ROOT = Path.home() / ".medsegbench"          # same location the DataConfigs use
SPLITS = ("train", "val", "test")
N_SAMPLE = 120                               # images used for the per-image feature sections

DATASETS = {                                 # resolution = what the arms actually train at
    "BUSI":    "busi_256.npz",
    "Kvasir":  "kvasir_256.npz",
    "MoNuSAC": "monusac_512.npz",
}
MONUSAC_CLASSES = {1: "Epithelial", 2: "Lymphocyte", 3: "Macrophage", 4: "Neutrophil", 5: "Ambiguous"}

# one accent hue + a de-emphasis grey, matching the palette used in notebook 02
ACCENT, MUTED = "#0173B2", "#B0B7BD"
COOL, WARM = "#0173B2", "#DE8F05"            # diverging poles for the correlation matrix

# np.load on an .npz is LAZY: every z["key"] re-decompresses the whole array. Materialise
# each one exactly once here (~10s, ~500MB) so every cell below is instant.
DATA = {}
for name, fn in DATASETS.items():
    z = np.load(ROOT / fn)
    imgs = z["train_images"]
    DATA[name] = {"lab": {s: z[f"{s}_label"] for s in SPLITS},
                  "img": imgs[:N_SAMPLE],
                  "shape": imgs.shape}
    print(f"{name:8s} " + ", ".join(f"{s}={DATA[name]['lab'][s].shape[0]}" for s in SPLITS)
          + f"   images {imgs.shape[1]}x{imgs.shape[2]}"
          + f", {1 if imgs.ndim == 3 else imgs.shape[3]} channel(s)")

def foreground(lab):
    """Boolean foreground mask. Excludes MoNuSAC's Ambiguous (5) everywhere."""
    return (lab > 0) & (lab != 5)

## 1. Sample size and image size

The basics: how much data there is, at what resolution, and what the label space looks like.


In [ ]:
rows = []
for name, d in DATA.items():
    shape = d["shape"]
    values = np.unique(np.concatenate([np.unique(d["lab"][s]) for s in SPLITS]))
    train_values = np.unique(d["lab"]["train"])
    rows.append({
        "dataset": name,
        "train": d["lab"]["train"].shape[0],
        "val":   d["lab"]["val"].shape[0],
        "test":  d["lab"]["test"].shape[0],
        "total": sum(d["lab"][s].shape[0] for s in SPLITS),
        "H x W": f"{shape[1]}x{shape[2]}",
        "ch": 1 if len(shape) == 3 else shape[3],
        "label values (all)":   str(values.tolist()),
        "label values (train)": str(train_values.tolist()),
    })
summary = pd.DataFrame(rows).set_index("dataset")
summary

Three things to carry into the report.

**The train splits differ by ~4x** (Kvasir 700, BUSI 452, MoNuSAC 188), so "how much data"
is a confound in any cross-dataset comparison. MoNuSAC is the most data-limited.

**MoNuSAC is 512px while the others are 256px.** That is deliberate (nuclei are a few
pixels across) but it means Dice is not measured on the same grid across datasets.

**MoNuSAC's label values include 5 (Ambiguous) only in test.** It has no train or val
examples, which is why the model is trained with 5 output channels and class 5 is
dropped at evaluation via `ignore_index`.


## 2. Class imbalance

For the binary sets this is just the foreground share. For MoNuSAC it is the share of
each nucleus type, which is where the real imbalance lives.


In [ ]:
# overall foreground share per dataset (train split)
rows = []
for name, d in DATA.items():
    fg = foreground(d["lab"]["train"])
    rows.append({"dataset": name,
                 "foreground %": 100 * fg.mean(),
                 "background %": 100 * (1 - fg.mean())})
pd.DataFrame(rows).set_index("dataset").round(2)

In [ ]:
# MoNuSAC per-class: pixel share, and how many images each class appears in
lab = DATA["MoNuSAC"]["lab"]["train"]
tot_px = lab.size
rows = []
for c, cname in MONUSAC_CLASSES.items():
    n_px = int((lab == c).sum())
    n_img = int(sum((lab[i] == c).any() for i in range(lab.shape[0])))
    rows.append({"class": cname, "pixel %": 100 * n_px / tot_px,
                 "images": n_img, "of images %": 100 * n_img / lab.shape[0]})
mono = pd.DataFrame(rows).set_index("class").round(2)
display(mono)

# one series -> one colour for every bar; a value-ramp here would just re-encode bar length
fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
sub = mono.loc[[c for c in mono.index if c != "Ambiguous"]]
for a, col, title in ((ax[0], "pixel %", "Share of all pixels"),
                      (ax[1], "of images %", "Share of images containing it")):
    bars = a.barh(sub.index, sub[col], color=ACCENT, height=0.6)
    a.bar_label(bars, fmt="%.1f%%", padding=3, fontsize=9)
    a.set_title(title, fontsize=10, loc="left")
    a.set_xlim(0, sub[col].max() * 1.25)
    a.invert_yaxis()
    for s in ("top", "right", "bottom"):
        a.spines[s].set_visible(False)
    a.tick_params(bottom=False, labelbottom=False)
fig.suptitle("MoNuSAC class imbalance (train split, Ambiguous excluded)", fontsize=11, x=0.01, ha="left")
plt.tight_layout(); plt.show()

## 3. How big is the target in each image?

The per-image foreground fraction. This is the single number that best explains why Dice
behaves so differently across the three: Dice rewards covering a large target, so a
dataset of big central objects scores high almost for free.


In [ ]:
# per-image foreground fraction, train split. Log x: the three span orders of magnitude.
fracs = {}
for name, d in DATA.items():
    lab = d["lab"]["train"]
    fracs[name] = foreground(lab).reshape(lab.shape[0], -1).mean(1)

fig, axes = plt.subplots(1, 3, figsize=(13, 3.2), sharex=True, sharey=True)
bins = np.logspace(-4, 0, 40)
for a, (name, v) in zip(axes, fracs.items()):
    a.hist(v, bins=bins, color=ACCENT, edgecolor="white", linewidth=0.5)
    a.axvline(np.median(v), color="#333", lw=1.2, ls="--")
    a.set_xscale("log")
    a.set_title(f"{name}   median {100*np.median(v):.1f}%", fontsize=10, loc="left")
    a.set_xlabel("foreground fraction of image")
    for s in ("top", "right"):
        a.spines[s].set_visible(False)
axes[0].set_ylabel("images")
fig.suptitle("Per-image target size (train split), dashed line = median",
             fontsize=11, x=0.01, ha="left")
plt.tight_layout(); plt.show()

pd.DataFrame({n: pd.Series(v).describe()[["mean", "std", "min", "50%", "max"]]
              for n, v in fracs.items()}).T.round(4)

## 4. One object or many?

Foreground fraction alone hides the structural difference. Kvasir and BUSI are one
object per image; MoNuSAC is dozens of small ones. Connected components make that explicit.


In [ ]:
rows = []
for name, d in DATA.items():
    lab = d["lab"]["train"][:N_SAMPLE]
    fg = foreground(lab)
    per_img = np.array([ndimage.label(fg[i])[1] for i in range(len(fg))])
    mean_obj_px = fg.reshape(len(fg), -1).sum(1) / np.maximum(per_img, 1)
    rows.append({"dataset": name,
                 "median objects/img": float(np.median(per_img)),
                 "max objects/img": int(per_img.max()),
                 "median object size (px)": float(np.median(mean_obj_px))})
pd.DataFrame(rows).set_index("dataset").round(1)

## 5. Correlation between image-level properties

Five properties per image, correlated within each dataset. Useful for spotting whether
target size is entangled with image appearance, which would mean a model could exploit
brightness as a shortcut instead of learning shape.


In [ ]:
def features(d):
    imgs, lab = d["img"], d["lab"]["train"][:N_SAMPLE]
    grey = imgs if imgs.ndim == 3 else imgs.mean(-1)
    fg = foreground(lab)
    n_obj = np.array([ndimage.label(fg[i])[1] for i in range(len(fg))])
    fg_px = fg.reshape(len(fg), -1).sum(1)
    return pd.DataFrame({
        "fg_fraction":  fg_px / fg[0].size,
        "n_objects":    n_obj,
        "obj_size":     fg_px / np.maximum(n_obj, 1),
        "img_mean":     grey.reshape(len(grey), -1).mean(1),
        "img_std":      grey.reshape(len(grey), -1).std(1),
    })

feats = {name: features(d) for name, d in DATA.items()}

# diverging map: two opposite hues with a neutral midpoint, since correlation is signed
from matplotlib.colors import LinearSegmentedColormap
DIVERGING = LinearSegmentedColormap.from_list("corr", [COOL, "#F2F2F2", WARM])

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for a, (name, df) in zip(axes, feats.items()):
    c = df.corr()
    a.imshow(c, cmap=DIVERGING, vmin=-1, vmax=1)
    a.set_xticks(range(len(c)), c.columns, rotation=45, ha="right", fontsize=8)
    a.set_yticks(range(len(c)), c.columns, fontsize=8)
    for i in range(len(c)):
        for j in range(len(c)):
            a.text(j, i, f"{c.iloc[i, j]:.2f}", ha="center", va="center", fontsize=7,
                   color="#111")                      # values printed: never colour-only
    a.set_title(name, fontsize=10, loc="left")
fig.suptitle(f"Correlation of image-level properties (train split, first {N_SAMPLE} images)",
             fontsize=11, x=0.01, ha="left")
plt.tight_layout(); plt.show()

## 6. Summary

What the three datasets contribute to the comparison:

| | BUSI | Kvasir | MoNuSAC |
|---|---|---|---|
| modality | grayscale ultrasound | RGB endoscopy | RGB H&E histopathology |
| task | binary lesion | binary polyp | 4-class nuclei |
| train images | 452 | 700 | 188 |
| target | one diffuse lesion | one large central polyp | dozens of tiny nuclei |
| boundaries | weak | clear | clear but tiny |

The three differ on modality, task type, target scale and dataset size at once, so they
are three separate tasks rather than a difficulty ranking. Any cross-dataset claim needs
to say which of those axes it is attributing a difference to.
